<a href="https://colab.research.google.com/github/vccf/Deep-Learning-Experiments/blob/working-demo-versions/YOLOv5Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I gonna use YOLOv5 and TPH-YOLOv5 to try to replicate and experiment on. The Dataset used will be VisDrone2019. In TPH-YOLOv5 article VisDrone2021 was used, but the article itself says it's the same dataset as VisDrone 2019. The goal of this notebook is to replicate the TPH-YOLOv5 article (within my limitations) and analyse it considering interpretability (using EigenCAM) and robustness (considering gaussian noise, blur, brightness and occlusion).

https://github.com/vccf/Deep-Learning-Experiments/

* https://github.com/ultralytics/yolov5
* https://github.com/cv516Buaa/tph-yolov5
* https://docs.ultralytics.com/yolov5/tutorials/train_custom_data
* https://docs.ultralytics.com/models/yolov5
* https://github.com/VisDrone/VisDrone-Dataset
* https://github.com/jacobgil/pytorch-grad-cam
* https://jacobgil.github.io/pytorch-gradcam-book/EigenCAM%20for%20YOLO5.html
* https://github.com/rigvedrs/YOLO-26-CAM


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
NVIDIA L4


In [ ]:
!pip install grad-cam #for interpretability

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 84.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44285 sha256=04f9d6c8999122ddad200bcb53ad63c81b659f22ed73d86363c2561a326c2f49
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
!pip show grad-cam

Name: grad-cam
Version: 1.5.5
Summary: Many Class Activation Map methods implemented in Pytorch for classification, segmentation, object detection and more
Home-page: https://github.com/jacobgil/pytorch-grad-cam
Author: Jacob Gildenblat
Author-email: jacob.gildenblat@gmail.com
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: matplotlib, numpy, opencv-python, Pillow, scikit-learn, torch, torchvision, tqdm, ttach
Required-by: 


In [ ]:
!git clone https://github.com/ultralytics/yolov5.git

Cloning into 'yolov5'...
remote: Enumerating objects: 18420, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 18420 (delta 58), reused 28 (delta 28), pack-reused 18335 (from 3)
Receiving objects: 100% (18420/18420), 17.53 MiB | 8.91 MiB/s, done.
Resolving deltas: 100% (12512/12512), done.


In [ ]:
%cd yolov5

!pip install -r requirements.txt

/content/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
#!python detect.py --weights yolov5l.pt --source data/images

In [ ]:
%cd ..

/content


In [ ]:
%mkdir datasets

In [ ]:
%cd datasets/

/content/datasets


In [ ]:
%mkdir VisDrone

In [ ]:
%cd VisDrone/

/content/datasets/VisDrone


In [ ]:
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-challenge.zip

--2026-07-05 06:01:16--  https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/9137a849-834e-4c54-b2d5-14d12384f10f?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-05T06%3A57%3A48Z&rscd=attachment%3B+filename%3DVisDrone2019-DET-train.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-05T05%3A57%3A37Z&ske=2026-07-05T06%3A57%3A48Z&sks=b&skv=2018-11-09&sig=ZbigNJ0r6DbbLMGuKFJnIU1s7a%2Fy7GbISUPoGVFfuZY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MzIzNDg3NywibmJmIjoxNzgzMjMxMjc3LCJwYXRoIjoicmVsZWF

In [ ]:
!unzip VisDrone2019-DET-train.zip
!unzip VisDrone2019-DET-val.zip
!unzip VisDrone2019-DET-test-dev.zip
!unzip VisDrone2019-DET-test-challenge.zip

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: VisDrone2019-DET-val/images/0000291_03801_d_0000887.jpg  
  inflating: VisDrone2019-DET-val/images/0000291_04001_d_0000888.jpg  
  inflating: VisDrone2019-DET-val/images/0000291_04201_d_0000889.jpg  
  inflating: VisDrone2019-DET-val/images/0000291_04401_d_0000890.jpg  
  inflating: VisDrone2019-DET-val/images/0000291_05001_d_0000892.jpg  
  inflating: VisDrone2019-DET-val/images/0000291_05201_d_0000893.jpg  
  inflating: VisDrone2019-DET-val/images/0000293_03601_d_0000940.jpg  
  inflating: VisDrone2019-DET-val/images/0000295_00000_d_0000021.jpg  
  inflating: VisDrone2019-DET-val/images/0000295_00200_d_0000022.jpg  
  inflating: VisDrone2019-DET-val/images/0000295_00600_d_0000024.jpg  
  inflating: VisDrone2019-DET-val/images/0000295_00800_d_0000025.jpg  
  inflating: VisDrone2019-DET-val/images/0000295_01000_d_0000026.jpg  
  inflating: VisDrone2019-DET-val/images/0000295_01200_d_0000027.jpg  
  inflating: VisDr

In [ ]:
!ls

VisDrone2019-DET-test-challenge      VisDrone2019-DET-train
VisDrone2019-DET-test-challenge.zip  VisDrone2019-DET-train.zip
VisDrone2019-DET-test-dev	     VisDrone2019-DET-val
VisDrone2019-DET-test-dev.zip	     VisDrone2019-DET-val.zip


In [ ]:
%cd /content/datasets/VisDrone/VisDrone2019-DET-train

/content/datasets/VisDrone/VisDrone2019-DET-train


In [ ]:
!ls

annotations  images


In [ ]:
%cd /content/datasets/VisDrone/VisDrone2019-DET-val

/content/datasets/VisDrone/VisDrone2019-DET-val


In [ ]:
!ls

annotations  images


In [ ]:
%cd /content/datasets/VisDrone/VisDrone2019-DET-test-dev/

/content/datasets/VisDrone/VisDrone2019-DET-test-dev


In [ ]:
!ls

annotations  images


In [ ]:
%cd /content/datasets/VisDrone

/content/datasets/VisDrone


In [ ]:
%mkdir -p /content/datasets/VisDrone/images/train
%mkdir -p /content/datasets/VisDrone/images/val
%mkdir -p /content/datasets/VisDrone/images/test
%mkdir -p /content/datasets/VisDrone/labels/train
%mkdir -p /content/datasets/VisDrone/labels/val
%mkdir -p /content/datasets/VisDrone/labels/test

In [ ]:
%cp /content/datasets/VisDrone/VisDrone2019-DET-train/images/* \
   /content/datasets/VisDrone/images/train/

%cp /content/datasets/VisDrone/VisDrone2019-DET-val/images/* \
   /content/datasets/VisDrone/images/val/

%cp /content/datasets/VisDrone/VisDrone2019-DET-test-dev/images/* \
   /content/datasets/VisDrone/images/test/

In [ ]:
%%writefile /content/yolov5/data/visdrone.yaml

path: /content/datasets/VisDrone

train: images/train
val: images/val
test : images/test

nc: 10

names:
  [
    0: pedestrian,
    1: people,
    2: bicycle,
    3: car,
    4: van,
    5: truck,
    6: tricycle,
    7: awning-tricycle,
    8: bus,
    9: motor
  ]

Writing /content/yolov5/data/visdrone.yaml


In [ ]:
!ls /content/yolov5/data/

Argoverse.yaml	      hyps		 images		  VisDrone.yaml
coco128-seg.yaml      ImageNet1000.yaml  Objects365.yaml  VOC.yaml
coco128.yaml	      ImageNet100.yaml	 scripts	  xView.yaml
coco.yaml	      ImageNet10.yaml	 SKU-110K.yaml
GlobalWheat2020.yaml  ImageNet.yaml	 visdrone.yaml


In [ ]:
!cat /content/yolov5/data/visdrone.yaml


path: /content/datasets/VisDrone

train: images/train
val: images/val
test : images/test

nc: 10

names:
  [
    0: pedestrian,
    1: people,
    2: bicycle,
    3: car,
    4: van,
    5: truck,
    6: tricycle,
    7: awning-tricycle,
    8: bus,
    9: motor
  ]


In [ ]:
import os
import cv2

def convert_visdrone(ann_dir, img_dir, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    files = [f for f in os.listdir(ann_dir) if f.endswith(".txt")]

    for ann_file in files:

        img_file = ann_file.replace(".txt", ".jpg")
        img_path = os.path.join(img_dir, img_file)

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        yolo_lines = []

        with open(os.path.join(ann_dir, ann_file)) as f:
            for line in f:

                parts = line.strip().split(",")

                if len(parts) < 8:
                    continue

                x = float(parts[0])
                y = float(parts[1])
                bw = float(parts[2])
                bh = float(parts[3])
                cls = int(parts[5])

                # ignore ignored regions
                if cls == 0:
                    continue

                xc = (x + bw/2) / w
                yc = (y + bh/2) / h

                bw /= w
                bh /= h

                yolo_cls = cls - 1

                yolo_lines.append(
                    f"{yolo_cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
                )

        with open(os.path.join(save_dir, ann_file), "w") as out:
            out.write("\n".join(yolo_lines))

In [ ]:
convert_visdrone(
    "/content/datasets/VisDrone/VisDrone2019-DET-train/annotations",
    "/content/datasets/VisDrone/VisDrone2019-DET-train/images",
    "/content/datasets/VisDrone/labels/train"
)

convert_visdrone(
    "/content/datasets/VisDrone/VisDrone2019-DET-val/annotations",
    "/content/datasets/VisDrone/VisDrone2019-DET-val/images",
    "/content/datasets/VisDrone/labels/val"
)

convert_visdrone(
    "/content/datasets/VisDrone/VisDrone2019-DET-test-dev/annotations",
    "/content/datasets/VisDrone/VisDrone2019-DET-test-dev/images",
    "/content/datasets/VisDrone/labels/test"
)

In [ ]:
!head -5 /content/datasets/VisDrone/labels/train/9999999_00879_d_0000403.txt

3 0.356000 0.907667 0.059000 0.174000
0 0.368750 0.774333 0.016500 0.038000
0 0.385500 0.638667 0.013000 0.034667
3 0.363500 0.609000 0.035000 0.083333
3 0.366000 0.533000 0.031000 0.074000


In [ ]:
!find /content/datasets/VisDrone -name "*.cache" -delete

In [ ]:
!find /content/datasets/VisDrone/images/train -name "*.jpg" | wc -l
!find /content/datasets/VisDrone/labels/train -name "*.txt" | wc -l

6471
6471


In [ ]:
!pwd

/content/datasets/VisDrone


In [ ]:
!ls /content/datasets/VisDrone/labels/train | head

0000002_00005_d_0000014.txt
0000002_00448_d_0000015.txt
0000003_00231_d_0000016.txt
0000007_04999_d_0000036.txt
0000007_05499_d_0000037.txt
0000007_05999_d_0000038.txt
0000008_00889_d_0000039.txt
0000008_01999_d_0000040.txt
0000008_02499_d_0000041.txt
0000008_02999_d_0000042.txt


In [ ]:
!find /content/datasets/VisDrone/images/train -name "*.jpg" | wc -l
!find /content/datasets/VisDrone/labels/train -name "*.txt" | wc -l

!find /content/datasets/VisDrone/images/val -name "*.jpg" | wc -l
!find /content/datasets/VisDrone/labels/val -name "*.txt" | wc -l

6471
6471
548
548


In [ ]:
!grep -R "," /content/datasets/VisDrone/labels/train | head
!grep -R "," /content/datasets/VisDrone/labels/val | head

In [ ]:
!grep -R "^10 " /content/datasets/VisDrone/labels/train | head

/content/datasets/VisDrone/labels/train/9999942_00000_d_0000146.txt:10 0.522500 0.881905 0.012143 0.024762
/content/datasets/VisDrone/labels/train/9999942_00000_d_0000146.txt:10 0.536429 0.881429 0.017143 0.018095
/content/datasets/VisDrone/labels/train/9999942_00000_d_0000146.txt:10 0.485714 0.740476 0.011429 0.025714
/content/datasets/VisDrone/labels/train/9999942_00000_d_0000146.txt:10 0.504643 0.702857 0.013571 0.024762
/content/datasets/VisDrone/labels/train/9999942_00000_d_0000146.txt:10 0.470000 0.624286 0.012857 0.029524
/content/datasets/VisDrone/labels/train/9999942_00000_d_0000146.txt:10 0.497500 0.509048 0.010714 0.020000
/content/datasets/VisDrone/labels/train/9999955_00000_d_0000410.txt:10 0.053214 0.636421 0.049286 0.046954
/content/datasets/VisDrone/labels/train/9999948_00000_d_0000030.txt:10 0.118929 0.381429 0.110714 0.092381
/content/datasets/VisDrone/labels/train/9999955_00000_d_0000238.txt:10 0.256786 0.645305 0.027857 0.049492
/content/datasets/VisDrone/labels/tra

In [ ]:
!awk '{print $1}' /content/datasets/VisDrone/labels/train/*.txt | sort -n | tail

10
10
10
10
10
10
10
10
10
10


In [ ]:
from pathlib import Path

for folder in [
    "/content/datasets/VisDrone/labels/train",
    "/content/datasets/VisDrone/labels/val",
    "/content/datasets/VisDrone/labels/test"
]:
    for f in Path(folder).glob("*.txt"):
        with open(f) as infile:
            lines = infile.readlines()

        lines = [line for line in lines if not line.startswith("10 ")]

        with open(f, "w") as outfile:
            outfile.writelines(lines)

print("Done")

Done


In [ ]:
!awk '{print $1}' /content/datasets/VisDrone/labels/train/*.txt | sort -n | tail

9
9
9
9
9
9
9
9
9
9


In [ ]:
!find /content/datasets/VisDrone -name "*.cache" -delete

In [ ]:
%cd /content/yolov5
#demo train
!python train.py --img 640 --optimizer Adam --batch 2 --epochs 1 --data /content/yolov5/data/visdrone.yaml --weights yolov5l.pt --patience=5 --name demo-train
#!python train.py --img 1152 --optimizer Adam --batch 8 --epochs 24 --data /content/yolov5/data/visdrone.yaml --weights yolov5l.pt --patience=5 --name visdrone_yolov5

In [ ]:
#from google.colab import files

#files.download("/content/yolov5/runs/train/visdrone_yolov5/weights/best.pt")

In [ ]:
!python val.py --weights /content/yolov5/runs/train/demo-train/weights/best.pt --img 640 --data /content/yolov5/data/visdrone.yaml --batch-size 2 --save-txt  --save-conf --task val --verbose --name demo-val
#!python val.py --weights /content/yolov5/runs/train/visdrone_yolov5/weights/best.pt --img 1280 --data /content/yolov5/data/visdrone.yaml --batch-size 8 --save-txt  --save-conf --task val --verbose --name visdrone_yolov5

python3: can't open file '/content/val.py': [Errno 2] No such file or directory


In [ ]:
!python val.py --weights /content/yolov5/runs/train/demo-train/weights/best.pt --data /content/yolov5/data/visdrone.yaml --img 640 --batch-size 2 --task test --verbose --save-txt --save-conf --name demo-test
#!python val.py --weights /content/yolov5/runs/train/visdrone_yolov5/weights/best.pt --data /content/yolov5/data/visdrone.yaml --img 1280 --task test --verbose --save-txt --save-conf --name visdrone_yolov5

In [ ]:
!python benchmarks.py

In [ ]:
!python detect.py --weights /content/yolov5/runs/train/demo-train/weights/best.pt --visualize
#!python detect.py --weights /content/yolov5/runs/train/visdrone_yolov5/weights/best.pt --visualize

In [ ]:
!ls /content/yolov5/runs/

In [ ]:
!ls /content/yolov5/runs/train/

In [ ]:
!ls /content/yolov5/runs/val/

In [ ]:
!ls /content/yolov5/runs/detect/

In [ ]:
!ls /content/yolov5/runs/detect/demo-val/
#!ls /content/yolov5/runs/detect/visdrone_yolov5/

In [ ]:
!ls /content/yolov5/runs/val/demo-val/
#!ls /content/yolov5/runs/val/visdrone_yolov5/

In [ ]:
!find runs/val -name "confusion_matrix.png"

In [ ]:
from IPython.display import Image, display

display(Image('/content/yolov5/runs/val/demo-val/confusion_matrix.png'))
#display(Image('/content/yolov5/runs/val/visdrone_yolov5/confusion_matrix.png'))

In [ ]:
display(Image('/content/yolov5/runs/val/demo-val/F1_curve.png'))
#display(Image('/content/yolov5/runs/val/visdrone_yolov5/F1_curve.png'))

In [ ]:
display(Image('/content/yolov5/runs/val/demo-val/PR_curve.png'))
#display(Image('/content/yolov5/runs/val/visdrone_yolov5/PR_curve.png'))

In [ ]:
display(Image('/content/yolov5/runs/val/demo-val/P_curve.png'))
#display(Image('/content/yolov5/runs/val/visdrone_yolov5/P_curve.png'))

In [ ]:
display(Image('/content/yolov5/runs/val/demo-val/R_curve.png'))
#display(Image('/content/yolov5/runs/val/visdrone_yolov5/R_curve.png')) #mAP, feature maps, saliency maps

In [ ]:
!ls /content/datasets/VisDrone/images/test

In [ ]:
import torch
from models.experimental import attempt_load

model = attempt_load('/content/yolov5/runs/train/demo-train/weights/best.pt', map_location="cuda:0")
print(type(model))
print(type(model.model))

In [ ]:
def pick_target_layer(model):
    """
    Handles both possible wrapping depths:
    - model.model.model[-2]  (Model wraps an nn.Sequential further)
    - model.model[-2]        (model.model IS already the nn.Sequential)
    """
    import torch.nn as nn

    inner = model.model
    if isinstance(inner, nn.Sequential):
        target_layer = inner[-2]
    else:
        # inner has its own .model attribute holding the Sequential
        target_layer = inner.model[-2]

    print(f"EigenCAM target layer: {type(target_layer).__name__}")
    return [target_layer]

In [ ]:
print(type(model))
print(type(model.model))

In [ ]:
#EigenCAM Visualization
#Produces 3 EigenCAM heatmap overlays showing which image regions the tph-yolov5 backbone/neck activates on most strongly.
#pip install grad-cam

import os
import sys
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

REPO_DIR    = '/content/yolov5'
WEIGHTS     = '/content/yolov5/runs/train/demo-train/weights/best.pt' #/content/yolov5/runs/train/visdrone_yolov5/weights/best.pt
VAL_IMG_DIR = '/content/datasets/VisDrone/images/val'     # folder of validation images
IMG_SIZE    = 640
DEVICE      = "0" #"cuda:0" if torch.cuda.is_available() else "cpu"
N_SAMPLES   = 3
SEED        = 0

OUT_DIR = Path('content/tph-yolov5/eval_outputs/eigencam')
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, os.path.abspath(REPO_DIR))
from models.experimental import attempt_load  # noqa: E402
from utils.augmentations import letterbox      # noqa: E402

def load_model():
    model = attempt_load(WEIGHTS, map_location=DEVICE)  # check utils/experimental.py
    model.eval()
    return model

def pick_target_layer(model):
    """
    The last block before the Detect() head is the conventional choice for YOLOv5 CAM work: it's the deepest feature map that still has spatial
    structure tied to the whole image, right before the multi-scale detection heads split things apart.
    tph-yolov5 inserts an extra TransformerBlock (the 'TPH' part) near the end of the neck -- model.model[-2] still lands on the right spot in the
    stock configs, but if you customized the yaml, print(model.model) once and confirm index -2 is a C3/TransformerBlock, not Detect() itself.
    """
    target_layer = model.model[-2] #target_layer = model.model.model[-2]
    return [target_layer]


def preprocess(img_path, img_size):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_letterboxed = letterbox(img_rgb, img_size, auto=True)[0]

    img_float = img_letterboxed.astype(np.float32) / 255.0
    tensor = torch.from_numpy(img_float).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    return tensor, img_float  # img_float used for the visualization overlay

class DetectWrapper(torch.nn.Module):
    """
    pytorch-grad-cam's base CAM class assumes forward() returns a single tensor (like classifier logits), so it can build placeholder target
    categories via argmax. YOLOv5's Model.forward() returns a tuple -- (inference_output, training_output) -- which breaks that assumption.
    This wrapper just unwraps the tuple so grad-cam sees a plain tensor. The target_layer object itself is untouched by this wrapping, since
    forward hooks fire on that module directly regardless of what calls it.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        outputs = self.model(x)
        if isinstance(outputs, (list, tuple)):
            return outputs[0]
        return outputs

def main():
    random.seed(SEED)
    model = load_model()
    target_layers = pick_target_layer(model)

    wrapped_model = DetectWrapper(model)
    cam = EigenCAM(model=wrapped_model, target_layers=target_layers)
    #cam = EigenCAM(model=model, target_layers=target_layers)
    # NOTE: pytorch-grad-cam >=1.4 dropped the `use_cuda` kwarg; device is
    # inferred from the model/input tensors instead.

    all_imgs = sorted(
        p for p in Path(VAL_IMG_DIR).glob("*")
        if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    if len(all_imgs) < N_SAMPLES:
        raise ValueError(f"Only found {len(all_imgs)} images in {VAL_IMG_DIR}")

    samples = random.sample(all_imgs, N_SAMPLES)

    for i, img_path in enumerate(samples, start=1):
        tensor, img_float = preprocess(img_path, IMG_SIZE)

        grayscale_cam = cam(input_tensor=tensor, targets=None)[0, :, :]
        cam_overlay = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

        out_path = OUT_DIR / f"eigencam_sample_{i}.jpg"
        cv2.imwrite(str(out_path), cv2.cvtColor(cam_overlay, cv2.COLOR_RGB2BGR))
        print(f"[ok] {img_path.name} -> {out_path}")


if __name__ == "__main__":
    main()

In [ ]:
display(Image('content/yolov5/eval_outputs/eigencam/eigencam_sample_1.jpg'))

In [ ]:
display(Image('content/yolov5/eval_outputs/eigencam/eigencam_sample_2.jpg'))

In [ ]:
display(Image('content/yolov5/eval_outputs/eigencam/eigencam_sample_3.jpg'))

In [ ]:
%%writefile /content/yolov5/03_corruptions.py

"""
03_corruptions.py

Reusable image corruption functions used to build robustness curves. Each function takes a BGR uint8 image (as read by cv2.imread) and a
severity level in {1, 2, 3, 4, 5} (1 = mildest, 5 = harshest), and returns a corrupted BGR uint8 image of the same shape.

Severity 0 is treated as "no corruption" (the original clean image) by the calling script, not by these functions.

These are deliberately simple, dependency-light implementations (numpy + opencv only) so they behave the same regardless of which augmentation
libraries are or aren't installed in your yolov5 environment.
"""

import numpy as np
import cv2

def gaussian_noise(img: np.ndarray, severity: int) -> np.ndarray:
    """Additive Gaussian noise in pixel space. Sigma scales with severity."""
    sigma_levels = {1: 5, 2: 10, 3: 20, 4: 35, 5: 55}
    sigma = sigma_levels[severity]

    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    noisy = img.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def gaussian_blur(img: np.ndarray, severity: int) -> np.ndarray:
    """Gaussian blur. Kernel size / sigma scale with severity."""
    kernel_levels = {1: 3, 2: 5, 3: 9, 4: 13, 5: 19}
    k = kernel_levels[severity]
    return cv2.GaussianBlur(img, (k, k), sigmaX=0)

def brightness(img: np.ndarray, severity: int, direction: str = "darken") -> np.ndarray:
    """
    Brightness shift via the HSV V-channel.
    direction="darken" simulates dusk/underexposure (the more common failure
    mode for aerial/drone imagery, which is tph-yolov5's usual domain);
    pass direction="brighten" if you want overexposure instead.
    """
    factor_levels = {1: 0.85, 2: 0.70, 3: 0.55, 4: 0.40, 5: 0.25}
    factor = factor_levels[severity]
    if direction == "brighten":
        factor = 1 + (1 - factor)  # e.g. 0.85 -> 1.15, 0.25 -> 1.75

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 2] = np.clip(hsv[..., 2] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

def occlusion(img: np.ndarray, severity: int, seed: int = None) -> np.ndarray:
    """
    Random rectangular occlusions (gray boxes) covering an increasing
    fraction of the image area. Boxes are placed at random, non-guaranteed
    non-overlapping positions -- at high severities some overlap is expected
    and realistic (mimics heavy clutter/obstruction).
    """
    area_frac_levels = {1: 0.03, 2: 0.07, 3: 0.12, 4: 0.20, 5: 0.30}
    area_frac = area_frac_levels[severity]
    n_boxes = severity + 1  # more, smaller boxes at higher severity

    rng = np.random.default_rng(seed)
    out = img.copy()
    h, w = img.shape[:2]
    total_area = h * w
    target_area_per_box = (total_area * area_frac) / n_boxes

    for _ in range(n_boxes):
        box_w = int(np.sqrt(target_area_per_box * rng.uniform(0.5, 2.0)))
        box_h = int(target_area_per_box / max(box_w, 1))
        box_w, box_h = min(box_w, w), min(box_h, h)

        x0 = rng.integers(0, max(w - box_w, 1))
        y0 = rng.integers(0, max(h - box_h, 1))
        gray_value = int(rng.integers(80, 160))  # mid-gray occluder
        out[y0:y0 + box_h, x0:x0 + box_w] = gray_value

    return out

CORRUPTIONS = {
    "gaussian_noise": gaussian_noise,
    "gaussian_blur": gaussian_blur,
    "brightness": brightness,
    "occlusion": occlusion,
}

In [ ]:
%%writefile /content/yolov5/04_robustness_evaluation.py

"""
04_robustness_evaluation.py

For each corruption type (gaussian_noise, gaussian_blur, brightness, occlusion) and severity level (0 = clean, 1..5 = increasing severity):

  1. Builds a corrupted copy of the validation image set.
  2. Points a temporary data.yaml at it (labels are symlinked, unchanged,
     since corruptions here don't move objects around).
  3. Runs tph-yolov5's own val.py evaluation to get mAP@0.5 and
     mAP@0.5:0.95, exactly as your training pipeline computes them.
  4. Collects everything into one long-form table.

Then produces:
  - 4 robustness plots (mAP vs. severity), one per corruption type
  - one CSV + one Markdown summary table (mAP@0.5 by corruption x severity)

This is the slow script (it re-runs validation 4 corruptions x 6 severities = 24 times, plus one shared clean baseline = 21 actual runs).
Expect this to take roughly 21x as long as a single val.py call on your dataset.
"""

import os
import sys
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
# Module names can't start with a digit for a normal `import`, so
# 03_corruptions.py is loaded explicitly by file path instead:
import importlib.util
_spec = importlib.util.spec_from_file_location(
    "corruptions", Path(__file__).parent / "03_corruptions.py"
)
corruptions_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(corruptions_mod)
CORRUPTIONS = corruptions_mod.CORRUPTIONS

REPO_DIR    = '/content/yolov5'
WEIGHTS     = '/content/yolov5/runs/train/demo-train/weights/best.pt' #/content/yolov5/runs/train/visdrone_yolov5/weights/best.pt
ORIG_DATA_YAML = '/content/yolov5/data/visdrone.yaml'
VAL_IMG_DIR = '/content/datasets/VisDrone/images/val'     # must match the "images" folder data.yaml's val: points to
VAL_LBL_DIR    = '/content/datasets/VisDrone/labels/val'     # sibling "labels" folder, standard YOLO layout
IMG_SIZE    = 640
DEVICE      = "0" #"cuda:0" if torch.cuda.is_available() else "cpu"
BATCH_SIZE     = 16
CONF_THRES     = 0.001
IOU_THRES      = 0.6
SEVERITIES     = [0, 1, 2, 3, 4, 5]      # 0 = clean baseline

TMP_ROOT = Path("./tmp_corrupted")
OUT_DIR  = Path('/content/yolov5/eval_outputs/robustness')
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, os.path.abspath(REPO_DIR))
import val as tph_val  # noqa: E402

if not labels_out.exists():
    os.symlink(Path(VAL_LBL_DIR).resolve(), labels_out)

def build_corrupted_dataset(name: str, severity: int) -> Path:
    """
    Returns a path to a data.yaml for this (corruption, severity) pair.
    Severity 0 just returns the original data.yaml -- no copying needed.
    """
    if severity == 0:
        return Path(ORIG_DATA_YAML)

    base = TMP_ROOT / f"{name}_sev{severity}"
    images_out = base / "images" / "val"
    labels_out = base / "labels" / "val"
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.parent.mkdir(parents=True, exist_ok=True)   # <-- add this line

    if not labels_out.exists():
        # These corruptions don't move/remove objects, so ground truth boxes
        # are untouched -- we just symlink the labels folder in at the
        # standard YOLO path (images/val -> labels/val) rather than copying.
        os.symlink(Path(VAL_LBL_DIR).resolve(), labels_out)

    corrupt_fn = CORRUPTIONS[name]
    src_images = sorted(
        p for p in Path(VAL_IMG_DIR).glob("*")
        if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    for img_path in src_images:
        out_path = images_out / img_path.name
        if out_path.exists():
            continue  # already generated in a previous run
        img = cv2.imread(str(img_path))
        corrupted = corrupt_fn(img, severity)
        cv2.imwrite(str(out_path), corrupted)

    # Build a data.yaml identical to the original but pointing val: here
    with open(ORIG_DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    data_cfg["val"] = str(images_out.resolve())
    tmp_yaml_path = base / "data.yaml"
    with open(tmp_yaml_path, "w") as f:
        yaml.safe_dump(data_cfg, f)

    return tmp_yaml_path


def evaluate(data_yaml_path: Path, run_name: str):
    results = tph_val.run(
        data=str(data_yaml_path),
        weights=WEIGHTS,
        batch_size=BATCH_SIZE,
        imgsz=IMG_SIZE,
        conf_thres=CONF_THRES,
        iou_thres=IOU_THRES,
        device=DEVICE,
        save_json=False,
        plots=False,
        project=str(OUT_DIR / "runs"),
        name=run_name,
        exist_ok=True,
        task="val",
    )
    # tph-yolov5's val.run returns something like:
    # (mp, mr, map50, map, box_loss, obj_loss, cls_loss), maps, t
    # >>> ADAPT: if your fork's return signature differs, adjust the
    # unpacking below (print(results) once to check order/shape).
    metrics = results[0]
    mp, mr, map50, map_5095 = metrics[0], metrics[1], metrics[2], metrics[3]
    return {"precision": mp, "recall": mr, "mAP50": map50, "mAP50_95": map_5095}


def main():
    rows = []

    # Run the clean baseline once and reuse it for every corruption type's
    # severity-0 point.
    print("=== Clean baseline (severity 0) ===")
    baseline = evaluate(Path(ORIG_DATA_YAML), "clean_baseline")
    for name in CORRUPTIONS:
        rows.append({"corruption": name, "severity": 0, **baseline})

    for name in CORRUPTIONS:
        for sev in tqdm([s for s in SEVERITIES if s != 0], desc=name):
            data_yaml_path = build_corrupted_dataset(name, sev)
            metrics = evaluate(data_yaml_path, f"{name}_sev{sev}")
            rows.append({"corruption": name, "severity": sev, **metrics})

    df = pd.DataFrame(rows).sort_values(["corruption", "severity"])
    df.to_csv(OUT_DIR / "mAP_results_long.csv", index=False)

    # --- Individual robustness plots -------------------------------------
    for name in CORRUPTIONS:
        sub = df[df["corruption"] == name].sort_values("severity")
        plt.figure(figsize=(6, 4.5))
        plt.plot(sub["severity"], sub["mAP50"], marker="o", label="mAP@0.5")
        plt.plot(sub["severity"], sub["mAP50_95"], marker="s", label="mAP@0.5:0.95")
        plt.xlabel("Corruption severity")
        plt.ylabel("mAP")
        plt.title(f"yolov5 robustness: {name.replace('_', ' ').title()}")
        plt.xticks(SEVERITIES)
        plt.ylim(0, 1)
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        out_path = OUT_DIR / f"{name}_robustness.png"
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"[ok] {out_path}")

    # --- Summary table (mAP@0.5, corruption x severity) -------------------
    pivot_50 = df.pivot(index="corruption", columns="severity", values="mAP50")
    pivot_50.columns = [f"severity_{c}" for c in pivot_50.columns]
    pivot_50.to_csv(OUT_DIR / "mAP_summary_table.csv")

    with open(OUT_DIR / "mAP_summary_table.md", "w") as f:
        f.write("# mAP@0.5 by corruption type and severity\n\n")
        f.write(pivot_50.round(4).to_markdown())
        f.write("\n\n# mAP@0.5:0.95 by corruption type and severity\n\n")
        pivot_5095 = df.pivot(index="corruption", columns="severity", values="mAP50_95")
        pivot_5095.columns = [f"severity_{c}" for c in pivot_5095.columns]
        f.write(pivot_5095.round(4).to_markdown())

    print(f"\n[ok] Summary tables written to {OUT_DIR}/mAP_summary_table.{{csv,md}}")
    print(pivot_50.round(4))


if __name__ == "__main__":
    main()

In [ ]:
!python 04_robustness_evaluation.py

In [ ]:
#For the 4 plots you already generated as PNGs
from IPython.display import Image, display

for name in ["gaussian_noise", "gaussian_blur", "brightness", "occlusion"]:
    display(Image(f"eval_outputs/robustness/{name}_robustness.png"))

In [ ]:
#For the summary table, load it back as a DataFrame so it renders as a proper table (nicer than reading raw CSV text):
import pandas as pd

df = pd.read_csv("eval_outputs/robustness/mAP_summary_table.csv", index_col=0)
df  # in a Colab cell, just having this as the last line renders it as a table

In [ ]:
#all 4 plots side-by-side in one grid instead of stacked one after another
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
names = ["gaussian_noise", "gaussian_blur", "brightness", "occlusion"]

for ax, name in zip(axes.flat, names):
    img = mpimg.imread(f"eval_outputs/robustness/{name}_robustness.png")
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(name.replace("_", " ").title())

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files
files.download('/content/yolov5/runs/val/demo-train/confusion_matrix.png')
files.download('/content/yolov5/runs/val/demo-train/F1_curve.png')
files.download('/content/yolov5/runs/val/demo-train/PR_curve.png')
files.download('/content/yolov5/runs/val/demo-train/P_curve.png')
files.download('/content/yolov5/runs/val/demo-train/R_curve.png')
files.download('/content/datasets/VisDrone/images/val/0000313_06601_d_0000469.jpg')
files.download('/content/datasets/VisDrone/images/val/0000330_00401_d_0000802.jpg')
files.download('/content/datasets/VisDrone/images/val/0000069_01878_d_0000005.jpg')
files.download('/content/tph-yolov5/eval_outputs/eigencam/eigencam_sample_1.jpg')
files.download('/content/tph-yolov5/eval_outputs/eigencam/eigencam_sample_2.jpg')
files.download('/content/tph-yolov5/eval_outputs/eigencam/eigencam_sample_3.jpg')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/yolov5/robustness_results', 'zip', '/content/yolov5/eval_outputs/robustness')
files.download('/content/yolov5/robustness_results.zip')

## Previous Training

In [ ]:
%cd /content/yolov5
#demo train
!python train.py --img 640 --optimizer Adam --batch 2 --epochs 2 --data /content/yolov5/data/visdrone.yaml --weights yolov5l.pt

/content/yolov5
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5l.pt, cfg=, data=/content/yolov5/data/visdrone.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=2, batch_size=4, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=F

In [ ]:
!python train.py --img 1280 --optimizer Adam --batch 8 --epochs 65 --data /content/yolov5/data/visdrone.yaml --weights yolov5l.pt --patience=5 --name visdrone_yolov5

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5l.pt, cfg=, data=/content/yolov5/data/visdrone.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=65, batch_size=8, imgsz=1280, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=Adam, sync_bn=False, workers=8, project=runs/train, name=visdrone_yolov5, exist_ok=False, quad=False, cos_lr=False, label_smoothing=0.0, patience=5, freeze=[0]

##Drafts EigenCAM

In [ ]:
"""import torch

model = torch.hub.load(
    '/content/yolov5',
    'custom',
    path='/content/yolov5/runs/train/visdrone_yolov5/weights/best.pt',
    source='local'
)

print(model.model)

In [ ]:
"""#last layer before object representation
target_layers = [model.model.model[23]]

images = [
    "/content/datasets/VisDrone/images/test/9999938_00000_d_0000506.jpg",
    "/content/datasets/VisDrone/images/test/9999938_00000_d_0000507.jpg",
    "/content/datasets/VisDrone/images/test/9999938_00000_d_0000508.jpg"
]

In [ ]:
"""import cv2
import matplotlib.pyplot as plt

plt.figure(figsize=(18,6))

for i, img_path in enumerate(images):

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(1,3,i+1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path.name)

plt.show()

In [ ]:
"""print(type(model)) #models.common.AutoShape
print(type(model.model)) #models.yolo.DetectionModel

In [ ]:
"""target_layer = model.model.model[23]
print(target_layer)

In [ ]:
"""from pytorch_grad_cam.utils.model_targets import *
import pytorch_grad_cam.utils.model_targets as mt
print(dir(mt))

In [ ]:
"""model = torch.hub.load(
    '/content/yolov5',
    'custom',
    path='/content/yolov5/runs/train/visdrone_yolov5/weights/best.pt',
    source='local'
)

model = model.model
model.eval()

In [ ]:
"""import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import random

from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
"""weights = "/content/yolov5/runs/train/visdrone_yolov5/weights/best.pt"

wrapper = torch.hub.load(
    "/content/yolov5",
    "custom",
    path=weights,
    source="local"
)

# Remove AutoShape
model = wrapper.model
model.eval()

In [ ]:
"""target_layers = [model.model[23]]

print(target_layers)

In [ ]:
"""img_dir = Path("/content/datasets/VisDrone/images/val")

images = random.sample(
    list(img_dir.glob("*.jpg")),
    3
)

images

In [ ]:
"""plt.figure(figsize=(18,6))

for i, img_path in enumerate(images):

    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.subplot(1,3,i+1)
    plt.imshow(image)
    plt.title(img_path.name)
    plt.axis("off")

plt.show()

In [ ]:
"""cam = EigenCAM(
    model=model,
    target_layers=target_layers
)

In [ ]:
"""import pytorch_grad_cam
print(pytorch_grad_cam.__file__)

/usr/local/lib/python3.12/dist-packages/pytorch_grad_cam/__init__.py


In [ ]:
"""import os
import pytorch_grad_cam

package_dir = os.path.dirname(pytorch_grad_cam.__file__)
print(package_dir)

/usr/local/lib/python3.12/dist-packages/pytorch_grad_cam


In [ ]:
#!grep -R "YOLOBoxScoreTarget" $package_dir

In [ ]:
"""import cv2
import torch

img = cv2.imread(str(images[0]))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

img = torch.from_numpy(img).float()
img = img.permute(2, 0, 1)
img = img.unsqueeze(0) / 255.0

with torch.no_grad():
    outputs = model(img)

print(type(outputs))

if isinstance(outputs, (list, tuple)):
    print("Number of outputs:", len(outputs))
    for i, out in enumerate(outputs):
        if torch.is_tensor(out):
            print(f"Output {i}: {out.shape}")
else:
    print(outputs.shape)